In [31]:
import pandas as pd
import numpy as np
from pulp import *
import warnings
warnings.filterwarnings('ignore')

In [32]:

def load_and_prepare_data():
    employees = pd.read_csv('employee_data.csv')
    training = pd.read_csv('training_and_development_data.csv')
    engagement = pd.read_csv('employee_engagement_survey_data.csv')

    active_employees = employees[employees['EmployeeStatus'] == 'Active'].copy()
    print(f"\nFiltered to {len(active_employees)} active employees (eligible for training)")

    active_employees = active_employees.merge(
        engagement,
        left_on='EmpID',
        right_on='Employee ID',
        how='left'
    )

    active_employees['Engagement Score'].fillna(3, inplace=True)
    active_employees['Satisfaction Score'].fillna(3, inplace=True)
    active_employees['Work-Life Balance Score'].fillna(3, inplace=True)

    eligible_employees = active_employees[active_employees['Current Employee Rating'] < 5].copy()

    print(f"Pre-filtered to {len(eligible_employees)} employees with improvement potential (rating < 5)")

    SAMPLE_SIZE = 100
    if len(eligible_employees) > SAMPLE_SIZE:
        eligible_employees = eligible_employees.sample(n=SAMPLE_SIZE, random_state=42)

    return eligible_employees, training

In [33]:
def extract_training_programs(training_df):

    training_programs = training_df.groupby('Training Program Name').agg({
        'Training Duration(Days)': 'mean',
        'Training Cost': 'mean',
        'Training Type': lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0]
    }).reset_index()

    training_programs.columns = ['Program', 'Duration', 'Cost', 'Type']

    outcome_effectiveness = training_df.groupby('Training Program Name')['Training Outcome'].apply(
        lambda x: sum(x.isin(['Passed', 'Completed'])) / len(x) if len(x) > 0 else 0.5
    ).reset_index()
    outcome_effectiveness.columns = ['Program', 'Effectiveness']

    training_programs = training_programs.merge(outcome_effectiveness, on='Program')

    print(f"\nIdentified {len(training_programs)} unique training programs")
    print("\nTraining Programs Available:")
    print(training_programs.to_string(index=False))

    return training_programs


In [34]:
def calculate_improvement_potential(employee, training_program):

    rating_gap = 5 - employee['Current Employee Rating']

    engagement_factor = employee['Engagement Score'] / 5.0
    satisfaction_factor = employee['Satisfaction Score'] / 5.0

    training_effectiveness = training_program['Effectiveness']

    expected_improvement = (
        rating_gap * 0.4 +
        engagement_factor * 0.25 +
        satisfaction_factor * 0.15 +
        training_effectiveness * 0.20
    )

    return max(0, expected_improvement)

In [35]:
def formulate_optimization_problem(employees_df, training_programs_df,
                                   budget=50000, max_capacity_per_program=20):

    prob = LpProblem("Training_Allocation_Optimization", LpMaximize)

    employees = employees_df['EmpID'].tolist()
    programs = training_programs_df['Program'].tolist()

    x = LpVariable.dicts("assign",
                         ((emp, prog) for emp in employees for prog in programs),
                         cat='Binary')

    print(f"\nCreated {len(employees) * len(programs)} decision variables")
    print(f"({len(employees)} employees x {len(programs)} training programs)")

    improvement_matrix = {}
    for _, emp_row in employees_df.iterrows():
        for _, prog_row in training_programs_df.iterrows():
            emp_id = emp_row['EmpID']
            prog_name = prog_row['Program']
            improvement = calculate_improvement_potential(emp_row, prog_row)
            improvement_matrix[(emp_id, prog_name)] = improvement

    prob += lpSum([improvement_matrix[(emp, prog)] * x[(emp, prog)]
                   for emp in employees for prog in programs]), "Total_Performance_Improvement"

    print("\nObjective Function: Maximize total expected improvement in employee ratings")


    cost_dict = dict(zip(training_programs_df['Program'], training_programs_df['Cost']))

    prob += lpSum([cost_dict[prog] * x[(emp, prog)]
                   for emp in employees for prog in programs]) <= budget, "Budget_Constraint"

    print(f"\nConstraint 1: Total training cost <= ${budget:,.2f}")


    for emp in employees:
        prob += lpSum([x[(emp, prog)] for prog in programs]) <= 1, f"One_Training_{emp}"

    print(f"Constraint 2: Each employee receives at most 1 training program")


    for prog in programs:
        prob += lpSum([x[(emp, prog)] for emp in employees]) <= max_capacity_per_program, \
                f"Capacity_{prog.replace(' ', '_').replace(',', '')}"

    print(f"Constraint 3: Maximum {max_capacity_per_program} employees per training program")


    total_employees = len(employees)
    gender_counts = employees_df['GenderCode'].value_counts()

    for gender in gender_counts.index:
        gender_employees = employees_df[employees_df['GenderCode'] == gender]['EmpID'].tolist()
        gender_proportion = len(gender_employees) / total_employees
        min_selection = max(1, int(gender_proportion * max_capacity_per_program * 0.5))

        prob += lpSum([x[(emp, prog)] for emp in gender_employees for prog in programs]) >= min_selection, \
                f"Fairness_{gender}"

    print(f"Constraint 4: Demographic fairness ensured (proportional gender representation)")

    return prob, x, employees, programs, improvement_matrix

In [36]:
def solve_optimization(prob):
    prob.solve(PULP_CBC_CMD(msg=0))

    status = LpStatus[prob.status]
    print(f"\nSolution Status: {status}")

    if status == 'Optimal':
        print("Optimal solution found!")
        return True
    else:
        print("No optimal solution found")
        return False

In [37]:
def extract_results(prob, x, employees, programs, employees_df,
                   training_programs_df, improvement_matrix):


    selected_assignments = []

    for emp in employees:
        for prog in programs:
            if x[(emp, prog)].varValue == 1:
                emp_data = employees_df[employees_df['EmpID'] == emp].iloc[0]
                prog_data = training_programs_df[training_programs_df['Program'] == prog].iloc[0]

                selected_assignments.append({
                    'Employee_ID': emp,
                    'Employee_Name': f"{emp_data['FirstName']} {emp_data['LastName']}",
                    'Current_Rating': emp_data['Current Employee Rating'],
                    'Engagement_Score': emp_data['Engagement Score'],
                    'Department': emp_data['DepartmentType'],
                    'Gender': emp_data['GenderCode'],
                    'Training_Program': prog,
                    'Training_Cost': prog_data['Cost'],
                    'Training_Duration': prog_data['Duration'],
                    'Expected_Improvement': improvement_matrix[(emp, prog)]
                })

    results_df = pd.DataFrame(selected_assignments)

    print(f"\nSOLUTION SUMMARY:")
    print(f"   Total employees selected for training: {len(results_df)}")
    print(f"   Total training cost: ${results_df['Training_Cost'].sum():,.2f}")
    print(f"   Total expected improvement: {results_df['Expected_Improvement'].sum():.2f}")
    print(f"   Average expected improvement per employee: {results_df['Expected_Improvement'].mean():.2f}")
    print(f"   Optimal objective value: {value(prob.objective):.4f}")

    print(f"\nTRAINING PROGRAM DISTRIBUTION:")
    program_distribution = results_df['Training_Program'].value_counts()
    for prog, count in program_distribution.items():
        print(f"   {prog}: {count} employees")

    print(f"\nDEMOGRAPHIC DISTRIBUTION:")
    gender_distribution = results_df['Gender'].value_counts()
    for gender, count in gender_distribution.items():
        print(f"   {gender}: {count} employees")

    print(f"\nDEPARTMENT DISTRIBUTION:")
    dept_distribution = results_df['Department'].value_counts().head(5)
    for dept, count in dept_distribution.items():
        print(f"   {dept}: {count} employees")

    print("SELECTED EMPLOYEES FOR TRAINING")

    display_df = results_df[[
        'Employee_ID', 'Employee_Name', 'Current_Rating',
        'Engagement_Score', 'Training_Program', 'Training_Cost',
        'Expected_Improvement'
    ]].sort_values('Expected_Improvement', ascending=False)

    results_df.to_csv('training_allocation_results.csv', index=False)
    print("\n" + display_df.to_string(index=False))
    return results_df


In [38]:
def analyze_impact(results_df, employees_df):
    total_improvement = results_df['Expected_Improvement'].sum()
    avg_improvement = results_df['Expected_Improvement'].mean()

    selected_emp_ids = results_df['Employee_ID'].tolist()
    selected_current_ratings = employees_df[employees_df['EmpID'].isin(selected_emp_ids)]['Current Employee Rating']

    current_avg_rating = selected_current_ratings.mean()
    expected_new_avg_rating = current_avg_rating + avg_improvement

    print(f"PERFORMANCE IMPROVEMENT PROJECTION:")
    print(f"Current avg rating of selected employees: {current_avg_rating:.2f}")
    print(f"Expected avg rating after training: {expected_new_avg_rating:.2f}")
    print(f"Overall improvement: +{avg_improvement:.2f} points ({(avg_improvement/current_avg_rating)*100:.1f}%)")

    total_cost = results_df['Training_Cost'].sum()
    cost_per_improvement_point = total_cost / total_improvement if total_improvement > 0 else 0

    print(f"\nRETURN ON INVESTMENT:")
    print(f"Total investment: ${total_cost:,.2f}")
    print(f"Total expected improvement points: {total_improvement:.2f}")
    print(f"Cost per improvement point: ${cost_per_improvement_point:,.2f}")

    high_impact = len(results_df[results_df['Expected_Improvement'] >= 2.0])
    medium_impact = len(results_df[(results_df['Expected_Improvement'] >= 1.0) &
                                   (results_df['Expected_Improvement'] < 2.0)])
    low_impact = len(results_df[results_df['Expected_Improvement'] < 1.0])

    print(f"\nIMPROVEMENT POTENTIAL BREAKDOWN:")
    print(f"High impact (>= 2.0 improvement): {high_impact} employees")
    print(f"Medium impact (1.0-2.0): {medium_impact} employees")
    print(f"Low impact (<1.0): {low_impact} employees")



In [39]:
def main():


    TRAINING_BUDGET = 50000
    MAX_CAPACITY_PER_PROGRAM = 20


    employees_df, training_df = load_and_prepare_data()
    training_programs_df = extract_training_programs(training_df)

    prob, x, employees, programs, improvement_matrix = formulate_optimization_problem(
        employees_df,
        training_programs_df,
        budget=TRAINING_BUDGET,
        max_capacity_per_program=MAX_CAPACITY_PER_PROGRAM
    )

    if solve_optimization(prob):
        results_df = extract_results(
            prob, x, employees, programs,
            employees_df, training_programs_df,
            improvement_matrix
        )

        analyze_impact(results_df, employees_df)

    else:
        print("\nOptimization failed - please check constraints and data")

In [40]:
main()


Filtered to 2458 active employees (eligible for training)
Pre-filtered to 2236 employees with improvement potential (rating < 5)

Identified 5 unique training programs

Training Programs Available:
               Program  Duration       Cost     Type  Effectiveness
  Communication Skills  2.950966 542.382229 Internal       0.537890
      Customer Service  2.955752 567.389451 Internal       0.527434
Leadership Development  3.033101 564.289251 Internal       0.527875
    Project Management  3.032841 563.732627 External       0.479475
      Technical Skills  2.906736 557.983782 External       0.438687

Created 500 decision variables
(100 employees x 5 training programs)

Objective Function: Maximize total expected improvement in employee ratings

Constraint 1: Total training cost <= $50,000.00
Constraint 2: Each employee receives at most 1 training program
Constraint 3: Maximum 20 employees per training program
Constraint 4: Demographic fairness ensured (proportional gender representatio